# Slice Misrouting Anomaly Detection Pipeline

This notebook is divided into clear parts.
Each part contains:
- a short markdown explanation
- the exact code for that part

Run the notebook top-to-bottom.

## Part 1: Setup, Constants, and Core Parsing Helpers

This part imports libraries, defines configuration/constants, and provides robust helper functions for:
- text cleaning
- numeric parsing (including decimal commas and scientific notation)
- column naming normalization

In [ ]:
"""
Anomaly detection pipeline for slice misrouting in 5G/6G network slicing datasets.

The script is intentionally notebook-friendly:
- It is organized in clear function blocks.
- You can run end-to-end as a script.
- You can also copy sections into an .ipynb notebook.

Main outputs:
- cleaned_dataset.csv
- engineered_dataset.csv
- final_model_dataset.csv
- plots/*.png
- results/*.csv and results/*.json
"""

from __future__ import annotations

import argparse
import io
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


RANDOM_SEED = 42
EPSILON = 1e-12


RENAME_MAP = {
    "Use Case Type": "use_case_type",
    "Packet Loss Budget": "packet_loss_budget",
    "Latency Budget (ns)": "latency_budget_ns",
    "Jitter Budget (ns)": "jitter_budget_ns",
    "Data Rate Budget (Gbps)": "data_rate_budget_gbps",
    "Required Mobility": "required_mobility",
    "Required Connectivity": "required_connectivity",
    "Slice Available Transfer Rate (Gbps)": "slice_transfer_rate_gbps",
    "Slice Latency (ns)": "slice_latency_ns",
    "Slice Packet Loss": "slice_packet_loss",
    "Slice Jitter (ns)": "slice_jitter_ns",
    "Slice Type": "slice_type",
    "Slice Handover": "slice_handover",
}


NUMERIC_COLUMNS = [
    "packet_loss_budget",
    "latency_budget_ns",
    "jitter_budget_ns",
    "data_rate_budget_gbps",
    "slice_transfer_rate_gbps",
    "slice_latency_ns",
    "slice_packet_loss",
    "slice_jitter_ns",
]


CATEGORICAL_COLUMNS = [
    "use_case_type",
    "required_mobility",
    "required_connectivity",
    "slice_type",
    "slice_handover",
]


DEFAULT_WEIGHTS = {
    "latency_violation": 2,
    "packet_loss_violation": 2,
    "jitter_violation": 1,
    "data_rate_violation": 1,
    "handover_mismatch": 1,
    "slice_type_mismatch": 2,
}


@dataclass
class PipelineConfig:
    csv_path: str
    output_dir: str = "."
    inject_synthetic: bool = False
    synthetic_fraction: float = 0.20
    random_seed: int = RANDOM_SEED
    drop_impossible_rows: bool = True
    severe_latency_ratio: float = 1.5
    severe_packet_loss_ratio: float = 1.5
    anomaly_min_violations: int = 2


def _normalize_col_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())


def _clean_text(value: object, to_lower: bool = True) -> Optional[str]:
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    if text == "" or text.lower() in {"nan", "none", "null", "na", "n/a"}:
        return np.nan
    return text.lower() if to_lower else text


def _parse_numeric_value(value: object) -> float:
    """
    Robust numeric parser for values like:
    - '1,00E-06'
    - '1.00E-06'
    - '1,234.56' / '1.234,56'
    - mixed whitespace strings
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)

    text = str(value).strip().replace("\u00a0", "").replace(" ", "")
    if text == "" or text.lower() in {"nan", "none", "null", "na", "n/a"}:
        return np.nan

    # Handle decimal comma in scientific notation: 1,00E-06 -> 1.00E-06
    if re.match(r"^[+-]?\d+,\d+[eE][+-]?\d+$", text):
        text = text.replace(",", ".")
    elif "," in text and "." not in text:
        # Could be decimal comma or thousands comma.
        if re.match(r"^[+-]?\d{1,3}(,\d{3})+$", text):
            text = text.replace(",", "")
        else:
            text = text.replace(",", ".")
    elif "," in text and "." in text:
        # Resolve ambiguous formatting by using right-most symbol as decimal marker.
        if text.rfind(",") > text.rfind("."):
            text = text.replace(".", "")
            text = text.replace(",", ".")
        else:
            text = text.replace(",", "")

    return pd.to_numeric(text, errors="coerce")



## Part 2: Data Loading and Defensive Cleaning

This part handles:
- CSV loading with delimiter detection
- exact column renaming
- `record_id` creation
- numeric conversion with coercion
- data quality checks (missing values, duplicates, impossible negatives)

In [ ]:
def load_data(csv_path: str) -> pd.DataFrame:
    """
    Load CSV defensively.
    Detect delimiter from header row if needed.
    Read all columns as strings first to prevent premature parsing failures.
    """
    csv_file = Path(csv_path)
    if not csv_file.exists():
        raise FileNotFoundError(f"CSV not found: {csv_file.resolve()}")

    first_line = csv_file.read_text(encoding="utf-8", errors="ignore").splitlines()[0]
    preferred_sep = ";" if first_line.count(";") > first_line.count(",") else ","

    df = pd.read_csv(csv_file, sep=preferred_sep, engine="python", dtype=str)
    if df.shape[1] == 1 and ";" in str(df.columns[0]):
        # Fallback if delimiter auto-choice fails.
        df = pd.read_csv(csv_file, sep=";", engine="python", dtype=str)

    print(f"[load_data] Loaded file: {csv_file}")
    print(f"[load_data] Shape: {df.shape}")
    print(f"[load_data] Columns: {list(df.columns)}")
    return df


def rename_and_clean_columns(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, object]]:
    """
    Rename columns using the required target names and standardize text columns.
    Adds record_id.
    """
    summary: Dict[str, object] = {}
    working = df.copy()
    working.columns = [str(c).strip() for c in working.columns]

    normalized_present = {_normalize_col_name(c): c for c in working.columns}
    rename_pairs = {}
    unresolved = []
    for original, target in RENAME_MAP.items():
        norm = _normalize_col_name(original)
        matched = normalized_present.get(norm)
        if matched is None:
            unresolved.append(original)
        else:
            rename_pairs[matched] = target

    if unresolved:
        raise ValueError(
            "Missing expected columns after normalization: "
            f"{unresolved}. Found columns: {list(working.columns)}"
        )

    working = working.rename(columns=rename_pairs)
    keep_cols = list(RENAME_MAP.values())
    working = working[keep_cols].copy()

    # Add stable record identifier.
    working.insert(0, "record_id", np.arange(1, len(working) + 1, dtype=np.int64))

    for col in CATEGORICAL_COLUMNS:
        working[col] = working[col].apply(lambda x: _clean_text(x, to_lower=True))

    summary["renamed_columns"] = rename_pairs
    summary["output_columns"] = list(working.columns)
    return working, summary


def convert_numeric_columns(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, object]]:
    """
    Convert numeric columns using robust parser with coercion.
    """
    working = df.copy()
    conversion_summary: Dict[str, object] = {"converted_columns": {}, "coerced_to_nan": {}}

    for col in NUMERIC_COLUMNS:
        if col not in working.columns:
            continue
        before_missing = int(working[col].isna().sum())
        working[col] = working[col].apply(_parse_numeric_value)
        after_missing = int(working[col].isna().sum())
        conversion_summary["converted_columns"][col] = "float"
        conversion_summary["coerced_to_nan"][col] = max(0, after_missing - before_missing)

    return working, conversion_summary


def run_data_quality_checks(
    df: pd.DataFrame, drop_impossible_rows: bool = True
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    """
    Report missingness, duplicates, impossible values.
    Optionally remove impossible rows.
    """
    working = df.copy()
    report: Dict[str, object] = {}

    report["missing_counts"] = working.isna().sum().to_dict()

    duplicate_mask = working.duplicated(subset=[c for c in working.columns if c != "record_id"])
    duplicate_count = int(duplicate_mask.sum())
    report["duplicate_rows"] = duplicate_count
    if duplicate_count > 0:
        working = working.loc[~duplicate_mask].copy()

    impossible_checks = {
        "latency_budget_ns": working["latency_budget_ns"] < 0,
        "jitter_budget_ns": working["jitter_budget_ns"] < 0,
        "data_rate_budget_gbps": working["data_rate_budget_gbps"] < 0,
        "packet_loss_budget": working["packet_loss_budget"] < 0,
        "slice_latency_ns": working["slice_latency_ns"] < 0,
        "slice_jitter_ns": working["slice_jitter_ns"] < 0,
        "slice_transfer_rate_gbps": working["slice_transfer_rate_gbps"] < 0,
        "slice_packet_loss": working["slice_packet_loss"] < 0,
    }

    impossible_flag = np.zeros(len(working), dtype=bool)
    invalid_counts = {}
    for col, cond in impossible_checks.items():
        cond_filled = cond.fillna(False)
        invalid_counts[col] = int(cond_filled.sum())
        impossible_flag = impossible_flag | cond_filled.to_numpy()

    working["impossible_values_flag"] = impossible_flag.astype(int)
    report["invalid_value_counts"] = invalid_counts
    report["impossible_row_count"] = int(working["impossible_values_flag"].sum())

    if drop_impossible_rows and report["impossible_row_count"] > 0:
        before = len(working)
        working = working.loc[working["impossible_values_flag"] == 0].copy()
        report["dropped_impossible_rows"] = before - len(working)
    else:
        report["dropped_impossible_rows"] = 0

    report["final_shape_after_quality_checks"] = working.shape
    return working, report



## Part 3: Full EDA (Exploratory Data Analysis)

This part includes:
- dataset inspection and summaries
- missingness and duplicate analysis
- distribution plots (matplotlib only)
- category frequency and crosstab heatmaps
- requirement-vs-slice comparisons
- correlation matrix and interpretation summary

In [ ]:
def _save_fig(fig: plt.Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def _plot_frequency(series: pd.Series, title: str, output_path: Path) -> None:
    counts = series.fillna("missing").value_counts(dropna=False)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(len(counts)), counts.values, color="#1f77b4", alpha=0.85, edgecolor="black")
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels([str(x) for x in counts.index], rotation=45, ha="right")
    ax.set_title(title)
    ax.set_ylabel("Count")
    _save_fig(fig, output_path)


def _plot_crosstab_heatmap(crosstab_df: pd.DataFrame, title: str, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    matrix = crosstab_df.to_numpy(dtype=float)
    im = ax.imshow(matrix, aspect="auto", cmap="Blues")
    fig.colorbar(im, ax=ax, label="Count")
    ax.set_xticks(np.arange(crosstab_df.shape[1]))
    ax.set_xticklabels([str(c) for c in crosstab_df.columns], rotation=45, ha="right")
    ax.set_yticks(np.arange(crosstab_df.shape[0]))
    ax.set_yticklabels([str(i) for i in crosstab_df.index])
    ax.set_title(title)
    ax.set_xlabel(crosstab_df.columns.name if crosstab_df.columns.name else "Columns")
    ax.set_ylabel(crosstab_df.index.name if crosstab_df.index.name else "Rows")
    _save_fig(fig, output_path)


def run_basic_eda(df: pd.DataFrame, plots_dir: Path, results_dir: Path) -> Dict[str, object]:
    """
    Full EDA requested:
    - inspection
    - missingness/duplicates
    - distribution plots
    - category plots and crosstabs
    - requirement vs slice comparison
    - correlation matrix
    - interpretation summary
    """
    eda_summary: Dict[str, object] = {}
    plots_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    print("\n[EDA] ===== BASIC INSPECTION =====")
    print(f"[EDA] shape: {df.shape}")
    print(f"[EDA] columns: {list(df.columns)}")

    info_buf = io.StringIO()
    df.info(buf=info_buf)
    info_text = info_buf.getvalue()
    print("\n[EDA] dataset info:\n", info_text)
    (results_dir / "dataset_info.txt").write_text(info_text, encoding="utf-8")

    numeric_summary = df[NUMERIC_COLUMNS].describe(include="all").T
    categorical_summary = df[CATEGORICAL_COLUMNS].describe(include="all").T
    print("\n[EDA] numeric summary:\n", numeric_summary)
    print("\n[EDA] categorical summary:\n", categorical_summary)
    numeric_summary.to_csv(results_dir / "numeric_summary.csv")
    categorical_summary.to_csv(results_dir / "categorical_summary.csv")

    requested_unique_cols = [
        "use_case_type",
        "slice_type",
        "required_mobility",
        "required_connectivity",
        "slice_handover",
    ]
    unique_values = {}
    for col in requested_unique_cols:
        values = sorted(df[col].dropna().astype(str).unique().tolist())
        unique_values[col] = values
        print(f"[EDA] unique values - {col}: {values}")
    eda_summary["unique_values"] = unique_values

    print("\n[EDA] ===== MISSINGNESS AND DUPLICATES =====")
    missing_counts = df.isna().sum().sort_values(ascending=False)
    duplicate_count = int(df.duplicated(subset=[c for c in df.columns if c != "record_id"]).sum())
    print("[EDA] missing value counts:\n", missing_counts)
    print(f"[EDA] duplicate rows (excluding record_id): {duplicate_count}")
    missing_counts.to_csv(results_dir / "missing_counts.csv", header=["missing_count"])

    print("\n[EDA] ===== DISTRIBUTIONS =====")
    distribution_cols = [
        "latency_budget_ns",
        "jitter_budget_ns",
        "packet_loss_budget",
        "data_rate_budget_gbps",
        "slice_latency_ns",
        "slice_jitter_ns",
        "slice_packet_loss",
        "slice_transfer_rate_gbps",
    ]
    for col in distribution_cols:
        series = df[col].dropna()
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].hist(series, bins=40, color="#2b8cbe", edgecolor="black", alpha=0.8)
        axes[0].set_title(f"Histogram - {col}")
        axes[0].set_xlabel(col)
        axes[0].set_ylabel("Count")
        axes[1].boxplot(series, vert=False)
        axes[1].set_title(f"Boxplot - {col}")
        _save_fig(fig, plots_dir / f"dist_{col}.png")

    print("\n[EDA] ===== CATEGORY RELATIONSHIPS =====")
    _plot_frequency(df["use_case_type"], "Use Case Type Frequency", plots_dir / "freq_use_case_type.png")
    _plot_frequency(df["slice_type"], "Slice Type Frequency", plots_dir / "freq_slice_type.png")

    use_case_slice_ct = pd.crosstab(df["use_case_type"], df["slice_type"], dropna=False)
    mobility_handover_ct = pd.crosstab(df["required_mobility"], df["slice_handover"], dropna=False)
    use_case_slice_ct.to_csv(results_dir / "crosstab_use_case_type_vs_slice_type.csv")
    mobility_handover_ct.to_csv(results_dir / "crosstab_required_mobility_vs_slice_handover.csv")
    _plot_crosstab_heatmap(
        use_case_slice_ct, "Use Case Type vs Slice Type (Count)", plots_dir / "ct_use_case_vs_slice_type.png"
    )
    _plot_crosstab_heatmap(
        mobility_handover_ct,
        "Required Mobility vs Slice Handover (Count)",
        plots_dir / "ct_mobility_vs_handover.png",
    )

    print("[EDA] use_case_type vs slice_type crosstab:\n", use_case_slice_ct)
    print("[EDA] required_mobility vs slice_handover crosstab:\n", mobility_handover_ct)

    print("\n[EDA] ===== REQUIREMENT VS SLICE CAPABILITY =====")
    comparison_specs = [
        ("latency_budget_ns", "slice_latency_ns", "latency"),
        ("jitter_budget_ns", "slice_jitter_ns", "jitter"),
        ("packet_loss_budget", "slice_packet_loss", "packet_loss"),
        ("data_rate_budget_gbps", "slice_transfer_rate_gbps", "data_rate"),
    ]
    comparison_rows = []
    for budget_col, slice_col, metric_name in comparison_specs:
        x = df[budget_col].to_numpy(dtype=float)
        y = df[slice_col].to_numpy(dtype=float)
        valid = np.isfinite(x) & np.isfinite(y)

        if metric_name == "data_rate":
            gap = x - y
            violation = y < x
        else:
            gap = y - x
            violation = y > x

        violation_rate = float(np.nanmean(violation.astype(float))) if valid.any() else np.nan
        comparison_rows.append(
            {
                "metric": metric_name,
                "budget_col": budget_col,
                "slice_col": slice_col,
                "budget_mean": float(np.nanmean(x)),
                "slice_mean": float(np.nanmean(y)),
                "gap_mean": float(np.nanmean(gap)),
                "violation_rate": violation_rate,
            }
        )

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].scatter(x[valid], y[valid], s=12, alpha=0.35, color="#1b7837")
        if valid.any():
            line_min = float(np.nanmin(np.concatenate([x[valid], y[valid]])))
            line_max = float(np.nanmax(np.concatenate([x[valid], y[valid]])))
            axes[0].plot([line_min, line_max], [line_min, line_max], "r--", linewidth=1)
        axes[0].set_xlabel(budget_col)
        axes[0].set_ylabel(slice_col)
        axes[0].set_title(f"{metric_name}: budget vs slice")
        axes[1].hist(gap[np.isfinite(gap)], bins=40, color="#d95f0e", edgecolor="black", alpha=0.8)
        axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
        axes[1].set_title(f"{metric_name}: gap distribution")
        axes[1].set_xlabel("Gap")
        axes[1].set_ylabel("Count")
        _save_fig(fig, plots_dir / f"requirement_vs_slice_{metric_name}.png")

    comparison_df = pd.DataFrame(comparison_rows)
    print("[EDA] requirement vs slice summary:\n", comparison_df)
    comparison_df.to_csv(results_dir / "requirement_vs_slice_summary.csv", index=False)

    print("\n[EDA] ===== CORRELATION ANALYSIS =====")
    corr_matrix = df[NUMERIC_COLUMNS].corr(numeric_only=True)
    corr_matrix.to_csv(results_dir / "numeric_correlation_matrix.csv")
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr_matrix.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    fig.colorbar(im, ax=ax, label="Pearson Correlation")
    ax.set_xticks(np.arange(len(corr_matrix.columns)))
    ax.set_xticklabels(corr_matrix.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(corr_matrix.index)))
    ax.set_yticklabels(corr_matrix.index)
    ax.set_title("Correlation Matrix (Numeric Columns)")
    _save_fig(fig, plots_dir / "correlation_heatmap.png")

    print("\n[EDA] ===== INTERPRETATION =====")
    violation_rates = comparison_df.set_index("metric")["violation_rate"].to_dict()
    raw_anomaly_signal = float(np.nanmean(list(violation_rates.values())))
    row_norm_ct = pd.crosstab(df["use_case_type"], df["slice_type"], normalize="index")
    mean_dominant_mapping = float(row_norm_ct.max(axis=1).mean()) if not row_norm_ct.empty else np.nan
    is_structured = bool((df["use_case_type"].nunique() <= 20) and (df["slice_type"].nunique() <= 20))

    interpretation = {
        "raw_violation_signal_mean": raw_anomaly_signal,
        "mean_dominant_use_case_to_slice_mapping": mean_dominant_mapping,
        "appears_structured_or_synthetic": is_structured,
        "natural_anomaly_evidence": bool(raw_anomaly_signal > 0.01),
    }
    eda_summary["interpretation"] = interpretation

    print(
        "[EDA][interpretation] Dataset appears mostly normal:"
        f" {'yes' if raw_anomaly_signal < 0.2 else 'mixed/no'} "
        f"(mean raw violation signal={raw_anomaly_signal:.4f})"
    )
    print(
        "[EDA][interpretation] use_case_type strongly determines slice_type:"
        f" {'yes' if mean_dominant_mapping >= 0.8 else 'partially/no'} "
        f"(mean dominant mapping strength={mean_dominant_mapping:.4f})"
    )
    print(
        "[EDA][interpretation] data looks synthetic/structured:"
        f" {'likely' if is_structured else 'not clearly'}"
    )
    print(
        "[EDA][interpretation] evidence of naturally occurring anomalies:"
        f" {'present' if interpretation['natural_anomaly_evidence'] else 'limited'}"
    )

    with (results_dir / "eda_interpretation.json").open("w", encoding="utf-8") as f:
        json.dump(interpretation, f, indent=2)

    return eda_summary



## Part 4: Feature Engineering and Rule-Based Labels

This part builds anomaly-oriented features:
- helper binary flags
- gap and ratio features
- violation flags
- handover mismatch
- expected slice mapping and slice mismatch
- aggregate anomaly features
- transparent rule-based `anomaly_label`

In [ ]:
def _map_binary_flag(value: object) -> float:
    """
    Map common telecom yes/no style strings to 0/1.
    Returns np.nan when unknown.
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return 1.0 if float(value) > 0 else 0.0

    text = _clean_text(value, to_lower=True)
    if pd.isna(text):
        return np.nan

    false_tokens = {
        "no",
        "n",
        "false",
        "0",
        "not supported",
        "unsupported",
        "not required",
        "disable",
        "disabled",
        "none",
        "static",
        "low",
    }
    true_tokens = {
        "yes",
        "y",
        "true",
        "1",
        "required",
        "supported",
        "enable",
        "enabled",
        "mobile",
        "high",
    }

    if text in false_tokens:
        return 0.0
    if text in true_tokens:
        return 1.0

    # phrase-level checks
    if "not" in text and ("support" in text or "require" in text):
        return 0.0
    if "support" in text or "require" in text:
        return 1.0

    return np.nan


def _safe_divide(numerator: pd.Series, denominator: pd.Series, epsilon: float = EPSILON) -> pd.Series:
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")
    result = np.where(np.abs(den) > epsilon, num / den, np.nan)
    return pd.Series(result, index=numerator.index)


def infer_expected_slice_mapping(df: pd.DataFrame) -> Dict[str, str]:
    """
    Data-driven expected mapping:
    expected slice for each use_case_type is the most frequent slice_type in cleaned data.
    """
    grouped = (
        df.groupby(["use_case_type", "slice_type"], dropna=True)
        .size()
        .reset_index(name="count")
        .sort_values(["use_case_type", "count", "slice_type"], ascending=[True, False, True])
    )
    mapping_df = grouped.drop_duplicates(subset=["use_case_type"], keep="first")
    return dict(zip(mapping_df["use_case_type"], mapping_df["slice_type"]))


def build_consistency_features(
    df: pd.DataFrame, weights: Optional[Dict[str, int]] = None
) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """
    Build helper, gap, ratio, violation, and aggregate anomaly features.
    """
    working = df.copy()
    used_weights = weights.copy() if weights is not None else DEFAULT_WEIGHTS.copy()

    # Helper binary columns
    working["mobility_required"] = working["required_mobility"].apply(_map_binary_flag)
    working["handover_supported"] = working["slice_handover"].apply(_map_binary_flag)
    working["connectivity_required"] = working["required_connectivity"].apply(_map_binary_flag)

    for col in ["mobility_required", "handover_supported", "connectivity_required"]:
        unresolved = int(working[col].isna().sum())
        if unresolved > 0:
            print(
                f"[build_consistency_features] Warning: {unresolved} rows in {col} "
                "could not be mapped; filling with 0."
            )
            working[col] = working[col].fillna(0)
        working[col] = working[col].astype(int)

    # Gap features (exact definitions requested)
    working["latency_gap_ns"] = working["slice_latency_ns"] - working["latency_budget_ns"]
    working["jitter_gap_ns"] = working["slice_jitter_ns"] - working["jitter_budget_ns"]
    working["packet_loss_gap"] = working["slice_packet_loss"] - working["packet_loss_budget"]
    working["data_rate_gap_gbps"] = working["data_rate_budget_gbps"] - working["slice_transfer_rate_gbps"]

    # Ratio features (safe division)
    # Assumption: denominator near zero => ratio undefined (set NaN).
    working["latency_ratio"] = _safe_divide(working["slice_latency_ns"], working["latency_budget_ns"])
    working["jitter_ratio"] = _safe_divide(working["slice_jitter_ns"], working["jitter_budget_ns"])
    working["packet_loss_ratio"] = _safe_divide(working["slice_packet_loss"], working["packet_loss_budget"])
    working["data_rate_ratio"] = _safe_divide(
        working["slice_transfer_rate_gbps"], working["data_rate_budget_gbps"]
    )

    # Violation flags
    working["latency_violation"] = (working["slice_latency_ns"] > working["latency_budget_ns"]).astype(int)
    working["jitter_violation"] = (working["slice_jitter_ns"] > working["jitter_budget_ns"]).astype(int)
    working["packet_loss_violation"] = (working["slice_packet_loss"] > working["packet_loss_budget"]).astype(int)
    working["data_rate_violation"] = (
        working["slice_transfer_rate_gbps"] < working["data_rate_budget_gbps"]
    ).astype(int)

    # Handover mismatch
    working["handover_mismatch"] = (
        (working["mobility_required"] == 1) & (working["handover_supported"] == 0)
    ).astype(int)

    # Expected slice mapping (data-driven)
    expected_map = infer_expected_slice_mapping(working)
    working["expected_slice_type"] = working["use_case_type"].map(expected_map)
    working["slice_type_mismatch"] = np.where(
        working["expected_slice_type"].isna() | working["slice_type"].isna(),
        0,
        (working["slice_type"] != working["expected_slice_type"]).astype(int),
    )

    violation_components = [
        "latency_violation",
        "jitter_violation",
        "packet_loss_violation",
        "data_rate_violation",
        "handover_mismatch",
        "slice_type_mismatch",
    ]
    working["violation_count"] = working[violation_components].sum(axis=1)

    # Weighted violation score
    working["weighted_violation_score"] = (
        used_weights["latency_violation"] * working["latency_violation"]
        + used_weights["packet_loss_violation"] * working["packet_loss_violation"]
        + used_weights["jitter_violation"] * working["jitter_violation"]
        + used_weights["data_rate_violation"] * working["data_rate_violation"]
        + used_weights["handover_mismatch"] * working["handover_mismatch"]
        + used_weights["slice_type_mismatch"] * working["slice_type_mismatch"]
    )

    working["consistency_flag"] = (working["violation_count"] == 0).astype(int)

    # Optional severity score using normalized positive gaps only.
    latency_severity = _safe_divide(working["latency_gap_ns"].clip(lower=0), working["latency_budget_ns"].abs() + EPSILON)
    jitter_severity = _safe_divide(working["jitter_gap_ns"].clip(lower=0), working["jitter_budget_ns"].abs() + EPSILON)
    packet_loss_severity = _safe_divide(
        working["packet_loss_gap"].clip(lower=0), working["packet_loss_budget"].abs() + EPSILON
    )
    data_rate_severity = _safe_divide(
        working["data_rate_gap_gbps"].clip(lower=0), working["data_rate_budget_gbps"].abs() + EPSILON
    )
    working["severity_score"] = (
        latency_severity.fillna(0)
        + jitter_severity.fillna(0)
        + packet_loss_severity.fillna(0)
        + data_rate_severity.fillna(0)
    )

    return working, expected_map


def build_rule_based_labels(
    df: pd.DataFrame,
    severe_latency_ratio: float = 1.5,
    severe_packet_loss_ratio: float = 1.5,
    anomaly_min_violations: int = 2,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    """
    Build transparent rule-based anomaly label.
    """
    working = df.copy()

    working["severe_latency_violation"] = (working["latency_ratio"] > severe_latency_ratio).astype(int)
    working["severe_packet_loss_violation"] = (
        working["packet_loss_ratio"] > severe_packet_loss_ratio
    ).astype(int)

    anomaly_condition = (
        (working["violation_count"] >= anomaly_min_violations)
        | (working["slice_type_mismatch"] == 1)
        | (working["handover_mismatch"] == 1)
        | (working["severe_latency_violation"] == 1)
        | (working["severe_packet_loss_violation"] == 1)
    )
    working["anomaly_label"] = anomaly_condition.astype(int)

    anomaly_rate = float(working["anomaly_label"].mean())
    class_balance = working["anomaly_label"].value_counts(dropna=False).to_dict()
    violation_counts = {
        "latency_violation": int(working["latency_violation"].sum()),
        "jitter_violation": int(working["jitter_violation"].sum()),
        "packet_loss_violation": int(working["packet_loss_violation"].sum()),
        "data_rate_violation": int(working["data_rate_violation"].sum()),
        "handover_mismatch": int(working["handover_mismatch"].sum()),
        "slice_type_mismatch": int(working["slice_type_mismatch"].sum()),
        "severe_latency_violation": int(working["severe_latency_violation"].sum()),
        "severe_packet_loss_violation": int(working["severe_packet_loss_violation"].sum()),
    }

    anomaly_patterns = (
        working.loc[working["anomaly_label"] == 1, [
            "latency_violation",
            "jitter_violation",
            "packet_loss_violation",
            "data_rate_violation",
            "handover_mismatch",
            "slice_type_mismatch",
            "severe_latency_violation",
            "severe_packet_loss_violation",
        ]]
        .value_counts()
        .head(10)
        .rename("count")
        .reset_index()
    )

    print("\n[rule_based_labels] anomaly_rate:", f"{anomaly_rate:.4%}")
    print("[rule_based_labels] class balance:", class_balance)
    print("[rule_based_labels] violation counts:", violation_counts)
    print("[rule_based_labels] top anomaly patterns:\n", anomaly_patterns)

    summary = {
        "anomaly_rate": anomaly_rate,
        "class_balance": class_balance,
        "violation_counts": violation_counts,
        "top_anomaly_patterns": anomaly_patterns.to_dict(orient="records"),
        "thresholds": {
            "severe_latency_ratio": severe_latency_ratio,
            "severe_packet_loss_ratio": severe_packet_loss_ratio,
            "anomaly_min_violations": anomaly_min_violations,
        },
    }
    return working, summary



## Part 5: Optional Synthetic Anomaly Injection + Model Feature Prep

This part adds optional synthetic anomalies for experimentation and defines feature sets for:
- unsupervised models
- supervised models

In [ ]:
def inject_synthetic_anomalies(
    clean_df: pd.DataFrame,
    injection_fraction: float = 0.20,
    random_seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    """
    Inject synthetic anomalies for experimentation.
    Strategy is intentionally simple and transparent.
    """
    if injection_fraction <= 0:
        result = clean_df.copy()
        result["synthetic_anomaly_injected"] = 0
        result["synthetic_anomaly_type"] = "none"
        return result

    rng = np.random.default_rng(random_seed)
    working = clean_df.copy()
    working["synthetic_anomaly_injected"] = 0
    working["synthetic_anomaly_type"] = "none"

    n_rows = len(working)
    n_inject = min(n_rows, max(1, int(np.floor(n_rows * injection_fraction))))
    selected_indices = rng.choice(working.index.to_numpy(), size=n_inject, replace=False)
    available_slice_types = [x for x in sorted(working["slice_type"].dropna().unique().tolist()) if str(x).strip()]

    mobility_flag = working["required_mobility"].apply(_map_binary_flag).fillna(0).astype(int)

    strategies = [
        "swap_slice_type",
        "latency_spike",
        "jitter_spike",
        "packet_loss_spike",
        "transfer_rate_drop",
        "disable_handover",
    ]

    for idx in selected_indices:
        order = list(rng.permutation(strategies))
        applied = False

        for strategy in order:
            if strategy == "swap_slice_type":
                current_type = working.at[idx, "slice_type"]
                candidates = [s for s in available_slice_types if s != current_type]
                if len(candidates) > 0:
                    working.at[idx, "slice_type"] = str(rng.choice(candidates))
                    working.at[idx, "synthetic_anomaly_injected"] = 1
                    working.at[idx, "synthetic_anomaly_type"] = "swap_slice_type"
                    applied = True
                    break

            elif strategy == "latency_spike":
                budget = working.at[idx, "latency_budget_ns"]
                current = working.at[idx, "slice_latency_ns"]
                if pd.notna(budget):
                    base = max(float(budget), 1.0)
                    working.at[idx, "slice_latency_ns"] = base * float(rng.uniform(1.6, 3.0))
                elif pd.notna(current):
                    working.at[idx, "slice_latency_ns"] = float(current) * float(rng.uniform(1.5, 2.5))
                else:
                    continue
                working.at[idx, "synthetic_anomaly_injected"] = 1
                working.at[idx, "synthetic_anomaly_type"] = "latency_spike"
                applied = True
                break

            elif strategy == "jitter_spike":
                budget = working.at[idx, "jitter_budget_ns"]
                current = working.at[idx, "slice_jitter_ns"]
                if pd.notna(budget):
                    base = max(float(budget), 1.0)
                    working.at[idx, "slice_jitter_ns"] = base * float(rng.uniform(1.6, 3.0))
                elif pd.notna(current):
                    working.at[idx, "slice_jitter_ns"] = float(current) * float(rng.uniform(1.5, 2.5))
                else:
                    continue
                working.at[idx, "synthetic_anomaly_injected"] = 1
                working.at[idx, "synthetic_anomaly_type"] = "jitter_spike"
                applied = True
                break

            elif strategy == "packet_loss_spike":
                budget = working.at[idx, "packet_loss_budget"]
                current = working.at[idx, "slice_packet_loss"]
                if pd.notna(budget):
                    base = max(float(budget), 1e-12)
                    working.at[idx, "slice_packet_loss"] = base * float(rng.uniform(1.6, 3.0))
                elif pd.notna(current):
                    working.at[idx, "slice_packet_loss"] = float(current) * float(rng.uniform(1.5, 2.5))
                else:
                    continue
                working.at[idx, "synthetic_anomaly_injected"] = 1
                working.at[idx, "synthetic_anomaly_type"] = "packet_loss_spike"
                applied = True
                break

            elif strategy == "transfer_rate_drop":
                budget = working.at[idx, "data_rate_budget_gbps"]
                if pd.notna(budget) and float(budget) > 0:
                    working.at[idx, "slice_transfer_rate_gbps"] = float(budget) * float(rng.uniform(0.05, 0.6))
                    working.at[idx, "synthetic_anomaly_injected"] = 1
                    working.at[idx, "synthetic_anomaly_type"] = "transfer_rate_drop"
                    applied = True
                    break

            elif strategy == "disable_handover":
                if int(mobility_flag.loc[idx]) == 1:
                    working.at[idx, "slice_handover"] = "no"
                    working.at[idx, "synthetic_anomaly_injected"] = 1
                    working.at[idx, "synthetic_anomaly_type"] = "disable_handover"
                    applied = True
                    break

        if not applied:
            # Fallback (always works unless missing value everywhere).
            current = working.at[idx, "slice_latency_ns"]
            if pd.notna(current):
                working.at[idx, "slice_latency_ns"] = float(current) * 2.0
                working.at[idx, "synthetic_anomaly_injected"] = 1
                working.at[idx, "synthetic_anomaly_type"] = "latency_spike_fallback"

    print(
        f"[inject_synthetic_anomalies] injected={int(working['synthetic_anomaly_injected'].sum())} "
        f"rows ({working['synthetic_anomaly_injected'].mean():.2%})"
    )
    print(
        "[inject_synthetic_anomalies] type distribution:\n",
        working["synthetic_anomaly_type"].value_counts(dropna=False),
    )
    return working


def prepare_model_data(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str], List[str], List[str]]:
    """
    Build final feature lists for unsupervised/supervised models.
    """
    working = df.copy()

    unsupervised_features = [
        "latency_gap_ns",
        "jitter_gap_ns",
        "packet_loss_gap",
        "data_rate_gap_gbps",
        "latency_ratio",
        "jitter_ratio",
        "packet_loss_ratio",
        "data_rate_ratio",
        "violation_count",
        "weighted_violation_score",
        "severity_score",
    ]

    supervised_numeric_features = [
        "packet_loss_budget",
        "latency_budget_ns",
        "jitter_budget_ns",
        "data_rate_budget_gbps",
        "slice_transfer_rate_gbps",
        "slice_latency_ns",
        "slice_packet_loss",
        "slice_jitter_ns",
        "mobility_required",
        "handover_supported",
        "connectivity_required",
        "latency_gap_ns",
        "jitter_gap_ns",
        "packet_loss_gap",
        "data_rate_gap_gbps",
        "latency_ratio",
        "jitter_ratio",
        "packet_loss_ratio",
        "data_rate_ratio",
        "latency_violation",
        "jitter_violation",
        "packet_loss_violation",
        "data_rate_violation",
        "handover_mismatch",
        "slice_type_mismatch",
        "violation_count",
        "weighted_violation_score",
        "severity_score",
    ]
    supervised_categorical_features = [
        "use_case_type",
        "required_mobility",
        "required_connectivity",
        "slice_type",
        "slice_handover",
        "expected_slice_type",
    ]

    for col in unsupervised_features + supervised_numeric_features + supervised_categorical_features:
        if col not in working.columns:
            raise ValueError(f"Required modeling column is missing: {col}")

    return (
        working,
        unsupervised_features,
        supervised_numeric_features,
        supervised_categorical_features,
    )



## Part 6: Modeling Functions

This part provides model training functions for:
- Isolation Forest (unsupervised)
- Random Forest classifier (supervised)
- evaluation helpers (confusion matrix, precision/recall/F1/ROC-AUC)

In [ ]:
def train_isolation_forest(
    df: pd.DataFrame,
    feature_columns: List[str],
    contamination: float = 0.15,
    random_seed: int = RANDOM_SEED,
) -> Dict[str, object]:
    """
    Unsupervised anomaly model (Isolation Forest).
    """
    x = df[feature_columns].copy()

    preprocess = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    x_proc = preprocess.fit_transform(x)

    model = IsolationForest(
        n_estimators=400,
        contamination=contamination,
        random_state=random_seed,
        n_jobs=-1,
    )
    model.fit(x_proc)

    # sklearn IF: predict => -1 (anomaly), 1 (normal)
    raw_pred = model.predict(x_proc)
    pred_label = np.where(raw_pred == -1, 1, 0).astype(int)

    # score_samples: lower means more abnormal. Convert so higher is more abnormal.
    anomaly_score = -model.score_samples(x_proc)
    threshold = float(np.quantile(anomaly_score, 1 - contamination))
    pred_by_threshold = (anomaly_score >= threshold).astype(int)

    return {
        "model": model,
        "preprocess": preprocess,
        "feature_columns": feature_columns,
        "pred_label": pred_label,
        "anomaly_score": anomaly_score,
        "threshold": threshold,
        "pred_by_threshold": pred_by_threshold,
    }


def train_supervised_model(
    df: pd.DataFrame,
    numeric_features: List[str],
    categorical_features: List[str],
    target_col: str = "anomaly_label",
    random_seed: int = RANDOM_SEED,
) -> Dict[str, object]:
    """
    Supervised model example using RandomForest.
    Includes one-hot encoding and numeric preprocessing.
    """
    if target_col not in df.columns:
        raise ValueError(f"Target column not found: {target_col}")

    x = df[numeric_features + categorical_features].copy()
    y = df[target_col].astype(int).copy()

    class_counts = y.value_counts()
    if y.nunique() < 2:
        raise ValueError(
            "anomaly_label contains a single class only. "
            "Supervised training requires at least two classes."
        )

    if int(class_counts.min()) < 2:
        # Defensive fallback for extreme imbalance (e.g., only 1 minority row).
        # Keep the rare class in train to guarantee model fit.
        print(
            "[train_supervised_model] Warning: extreme class imbalance detected "
            f"({class_counts.to_dict()}). Using custom split without stratification."
        )
        rng = np.random.default_rng(random_seed)
        minority_label = class_counts.idxmin()
        minority_idx = y[y == minority_label].index.to_numpy()
        majority_idx = y[y != minority_label].index.to_numpy()
        rng.shuffle(majority_idx)

        test_n = max(1, int(0.25 * len(y)))
        test_n = min(test_n, max(1, len(majority_idx) - 1))
        test_idx = majority_idx[:test_n]
        train_idx = np.concatenate([minority_idx, majority_idx[test_n:]])

        x_train, x_test = x.loc[train_idx], x.loc[test_idx]
        y_train, y_test = y.loc[train_idx], y.loc[test_idx]
    else:
        x_train, x_test, y_train, y_test = train_test_split(
            x,
            y,
            test_size=0.25,
            random_state=random_seed,
            stratify=y,
        )

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ]
    )

    model = RandomForestClassifier(
        n_estimators=400,
        random_state=random_seed,
        n_jobs=-1,
        class_weight="balanced",
        min_samples_leaf=1,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", model),
        ]
    )

    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)

    if hasattr(pipeline.named_steps["model"], "predict_proba"):
        y_proba = pipeline.predict_proba(x_test)[:, 1]
    else:
        y_proba = None

    return {
        "pipeline": pipeline,
        "x_train": x_train,
        "x_test": x_test,
        "y_train": y_train,
        "y_test": y_test,
        "y_pred": y_pred,
        "y_proba": y_proba,
    }


def evaluate_supervised_model(y_true: pd.Series, y_pred: np.ndarray, y_proba: Optional[np.ndarray]) -> Dict[str, object]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None and len(np.unique(y_true)) > 1:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_proba))
    else:
        metrics["roc_auc"] = np.nan
    return metrics


def evaluate_unsupervised_model(y_true: pd.Series, y_pred: np.ndarray) -> Dict[str, object]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    detected_pct = float(np.mean(y_pred) * 100.0)
    fpr = float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "detected_anomaly_percentage": detected_pct,
        "false_positive_rate": fpr,
        "confusion_matrix": cm.tolist(),
    }



## Part 7: Evaluation, Interpretation, Saving Outputs, and Orchestration

This part includes:
- confusion matrix plotting
- feature importance extraction
- anomalous-row interpretation helper
- CSV/JSON output saving
- `run_pipeline()` main orchestration function

In [ ]:
def _plot_confusion_matrix(cm: np.ndarray, labels: List[str], title: str, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    _save_fig(fig, output_path)


def extract_feature_importance(
    supervised_pipeline: Pipeline,
    output_csv: Path,
    output_plot: Path,
    top_n: int = 20,
) -> pd.DataFrame:
    model = supervised_pipeline.named_steps["model"]
    preprocess = supervised_pipeline.named_steps["preprocess"]

    if not hasattr(model, "feature_importances_"):
        print("[feature_importance] Model has no feature_importances_. Skipping.")
        return pd.DataFrame(columns=["feature", "importance"])

    feature_names = preprocess.get_feature_names_out()
    importances = model.feature_importances_
    fi_df = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    fi_df.to_csv(output_csv, index=False)

    top_df = fi_df.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top_df["feature"], top_df["importance"], color="#2c7fb8")
    ax.set_title(f"Top {top_n} Feature Importances (Random Forest)")
    ax.set_xlabel("Importance")
    _save_fig(fig, output_plot)
    return fi_df


def explain_rule_trigger(row: pd.Series, min_violations: int = 2) -> str:
    triggers = []
    if row.get("violation_count", 0) >= min_violations:
        triggers.append(f"violation_count>={min_violations}")
    if row.get("slice_type_mismatch", 0) == 1:
        triggers.append("slice_type_mismatch")
    if row.get("handover_mismatch", 0) == 1:
        triggers.append("handover_mismatch")
    if row.get("severe_latency_violation", 0) == 1:
        triggers.append("severe_latency_violation")
    if row.get("severe_packet_loss_violation", 0) == 1:
        triggers.append("severe_packet_loss_violation")
    return ", ".join(triggers) if triggers else "no_rule_triggered"


def print_anomaly_examples(
    df: pd.DataFrame,
    results_dir: Path,
    min_violations: int = 2,
    n_examples: int = 10,
) -> pd.DataFrame:
    cols = [
        "record_id",
        "use_case_type",
        "slice_type",
        "latency_budget_ns",
        "jitter_budget_ns",
        "packet_loss_budget",
        "data_rate_budget_gbps",
        "slice_latency_ns",
        "slice_jitter_ns",
        "slice_packet_loss",
        "slice_transfer_rate_gbps",
        "latency_violation",
        "jitter_violation",
        "packet_loss_violation",
        "data_rate_violation",
        "handover_mismatch",
        "slice_type_mismatch",
        "violation_count",
        "weighted_violation_score",
        "anomaly_label",
    ]
    available_cols = [c for c in cols if c in df.columns]
    subset = df.loc[df["anomaly_label"] == 1, available_cols].copy()
    subset = subset.head(n_examples).copy()

    if "iforest_pred" in df.columns:
        subset["iforest_pred"] = df.loc[subset.index, "iforest_pred"].values
    if "iforest_pred_threshold" in df.columns:
        subset["iforest_pred_threshold"] = df.loc[subset.index, "iforest_pred_threshold"].values
    if "supervised_pred" in df.columns:
        subset["supervised_pred"] = df.loc[subset.index, "supervised_pred"].values

    rule_src = df.loc[subset.index]
    subset["rule_trigger"] = rule_src.apply(
        lambda r: explain_rule_trigger(r, min_violations=min_violations), axis=1
    ).values

    print("\n[interpretation] example anomalous rows:\n", subset)
    subset.to_csv(results_dir / "anomaly_examples.csv", index=False)
    return subset


def save_outputs(
    cleaned_df: pd.DataFrame,
    engineered_df: pd.DataFrame,
    final_model_df: pd.DataFrame,
    output_dir: Path,
) -> None:
    cleaned_df.to_csv(output_dir / "cleaned_dataset.csv", index=False)
    engineered_df.to_csv(output_dir / "engineered_dataset.csv", index=False)
    final_model_df.to_csv(output_dir / "final_model_dataset.csv", index=False)
    print(f"[save_outputs] cleaned_dataset.csv -> {output_dir / 'cleaned_dataset.csv'}")
    print(f"[save_outputs] engineered_dataset.csv -> {output_dir / 'engineered_dataset.csv'}")
    print(f"[save_outputs] final_model_dataset.csv -> {output_dir / 'final_model_dataset.csv'}")


def run_pipeline(config: PipelineConfig) -> Dict[str, object]:
    base_output = Path(config.output_dir).resolve()
    plots_dir = base_output / "plots"
    results_dir = base_output / "results"
    plots_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    # 1) Load + clean
    raw_df = load_data(config.csv_path)
    renamed_df, rename_summary = rename_and_clean_columns(raw_df)
    numeric_df, numeric_summary = convert_numeric_columns(renamed_df)
    cleaned_df, quality_report = run_data_quality_checks(
        numeric_df, drop_impossible_rows=config.drop_impossible_rows
    )

    with (results_dir / "cleaning_summary.json").open("w", encoding="utf-8") as f:
        json.dump(
            {
                "rename_summary": rename_summary,
                "numeric_summary": numeric_summary,
                "quality_report": quality_report,
            },
            f,
            indent=2,
        )

    print("\n[cleaning] rename summary:", rename_summary)
    print("[cleaning] numeric conversion summary:", numeric_summary)
    print("[cleaning] data quality report:", quality_report)

    # 2) EDA on cleaned data
    eda_summary = run_basic_eda(cleaned_df, plots_dir=plots_dir, results_dir=results_dir)

    # 3-5) Feature engineering + rule labels (+ optional synthetic injection)
    base_for_features = cleaned_df.copy()
    if config.inject_synthetic:
        base_for_features = inject_synthetic_anomalies(
            base_for_features,
            injection_fraction=config.synthetic_fraction,
            random_seed=config.random_seed,
        )
    else:
        base_for_features["synthetic_anomaly_injected"] = 0
        base_for_features["synthetic_anomaly_type"] = "none"

    engineered_df, expected_mapping = build_consistency_features(base_for_features)
    labeled_df, labeling_summary = build_rule_based_labels(
        engineered_df,
        severe_latency_ratio=config.severe_latency_ratio,
        severe_packet_loss_ratio=config.severe_packet_loss_ratio,
        anomaly_min_violations=config.anomaly_min_violations,
    )

    pd.Series(expected_mapping, name="expected_slice_type").to_csv(
        results_dir / "expected_slice_mapping.csv", header=True
    )

    with (results_dir / "labeling_summary.json").open("w", encoding="utf-8") as f:
        json.dump(labeling_summary, f, indent=2)

    # 6) Prepare modeling dataset
    model_df, if_features, sup_num_features, sup_cat_features = prepare_model_data(labeled_df)

    # 7A) Isolation Forest
    # Contamination is set relative to rule-based anomaly rate, clipped to practical bounds.
    inferred_contamination = float(model_df["anomaly_label"].mean())
    if_contamination = float(np.clip(inferred_contamination, 0.05, 0.35))
    iforest_artifacts = train_isolation_forest(
        model_df,
        feature_columns=if_features,
        contamination=if_contamination,
        random_seed=config.random_seed,
    )
    model_df["iforest_pred"] = iforest_artifacts["pred_label"]
    model_df["iforest_score"] = iforest_artifacts["anomaly_score"]
    model_df["iforest_pred_threshold"] = iforest_artifacts["pred_by_threshold"]

    # 7B) Supervised model
    supervised_artifacts = train_supervised_model(
        model_df,
        numeric_features=sup_num_features,
        categorical_features=sup_cat_features,
        target_col="anomaly_label",
        random_seed=config.random_seed,
    )

    # Add test-set predictions back into final frame for interpretation.
    model_df["supervised_pred"] = np.nan
    model_df["supervised_pred_proba"] = np.nan
    x_test_idx = supervised_artifacts["x_test"].index
    model_df.loc[x_test_idx, "supervised_pred"] = supervised_artifacts["y_pred"]
    if supervised_artifacts["y_proba"] is not None:
        model_df.loc[x_test_idx, "supervised_pred_proba"] = supervised_artifacts["y_proba"]
    model_df["supervised_pred"] = model_df["supervised_pred"].fillna(0).astype(int)

    # 8) Evaluation
    sup_metrics = evaluate_supervised_model(
        supervised_artifacts["y_test"],
        supervised_artifacts["y_pred"],
        supervised_artifacts["y_proba"],
    )
    unsup_metrics = evaluate_unsupervised_model(model_df["anomaly_label"], model_df["iforest_pred_threshold"])

    with (results_dir / "supervised_metrics.json").open("w", encoding="utf-8") as f:
        json.dump(sup_metrics, f, indent=2)
    with (results_dir / "isolation_forest_metrics.json").open("w", encoding="utf-8") as f:
        json.dump(unsup_metrics, f, indent=2)

    cm_sup = np.array(sup_metrics["confusion_matrix"], dtype=int)
    cm_unsup = np.array(unsup_metrics["confusion_matrix"], dtype=int)
    _plot_confusion_matrix(
        cm_sup,
        labels=["normal(0)", "anomaly(1)"],
        title="Supervised Confusion Matrix",
        output_path=plots_dir / "cm_supervised.png",
    )
    _plot_confusion_matrix(
        cm_unsup,
        labels=["normal(0)", "anomaly(1)"],
        title="Isolation Forest Confusion Matrix",
        output_path=plots_dir / "cm_isolation_forest.png",
    )

    comparison_table = pd.DataFrame(
        [
            {
                "model": "RandomForest(supervised)",
                "accuracy": sup_metrics.get("accuracy"),
                "precision": sup_metrics.get("precision"),
                "recall": sup_metrics.get("recall"),
                "f1_score": sup_metrics.get("f1_score"),
                "roc_auc": sup_metrics.get("roc_auc"),
            },
            {
                "model": "IsolationForest(unsupervised)",
                "accuracy": np.nan,
                "precision": unsup_metrics.get("precision"),
                "recall": unsup_metrics.get("recall"),
                "f1_score": unsup_metrics.get("f1_score"),
                "roc_auc": np.nan,
            },
        ]
    )
    comparison_table.to_csv(results_dir / "model_comparison.csv", index=False)

    print("\n[evaluation] supervised metrics:")
    print(json.dumps(sup_metrics, indent=2))
    print("\n[evaluation] isolation forest metrics:")
    print(json.dumps(unsup_metrics, indent=2))
    print("\n[evaluation] model comparison table:\n", comparison_table)

    # 9) Interpretation examples + feature importance
    anomaly_examples = print_anomaly_examples(
        model_df, results_dir=results_dir, min_violations=config.anomaly_min_violations, n_examples=12
    )

    fi_df = extract_feature_importance(
        supervised_artifacts["pipeline"],
        output_csv=results_dir / "feature_importance_random_forest.csv",
        output_plot=plots_dir / "feature_importance_random_forest.png",
        top_n=20,
    )
    if not fi_df.empty:
        print("\n[interpretation] top feature importance rows:\n", fi_df.head(10))

    # 6/10) Final column ordering for deliverable dataset.
    final_columns = [
        "record_id",
        "use_case_type",
        "packet_loss_budget",
        "latency_budget_ns",
        "jitter_budget_ns",
        "data_rate_budget_gbps",
        "required_mobility",
        "required_connectivity",
        "slice_transfer_rate_gbps",
        "slice_latency_ns",
        "slice_packet_loss",
        "slice_jitter_ns",
        "slice_type",
        "slice_handover",
        "mobility_required",
        "handover_supported",
        "connectivity_required",
        "latency_gap_ns",
        "jitter_gap_ns",
        "packet_loss_gap",
        "data_rate_gap_gbps",
        "latency_ratio",
        "jitter_ratio",
        "packet_loss_ratio",
        "data_rate_ratio",
        "latency_violation",
        "jitter_violation",
        "packet_loss_violation",
        "data_rate_violation",
        "handover_mismatch",
        "expected_slice_type",
        "slice_type_mismatch",
        "violation_count",
        "weighted_violation_score",
        "consistency_flag",
        "severity_score",
        "severe_latency_violation",
        "severe_packet_loss_violation",
        "anomaly_label",
        "synthetic_anomaly_injected",
        "synthetic_anomaly_type",
        "iforest_pred",
        "iforest_score",
        "iforest_pred_threshold",
        "supervised_pred",
        "supervised_pred_proba",
    ]
    final_columns = [c for c in final_columns if c in model_df.columns]
    final_model_df = model_df[final_columns].copy()

    save_outputs(cleaned_df=cleaned_df, engineered_df=labeled_df, final_model_df=final_model_df, output_dir=base_output)

    return {
        "cleaned_df": cleaned_df,
        "engineered_df": labeled_df,
        "final_model_df": final_model_df,
        "eda_summary": eda_summary,
        "labeling_summary": labeling_summary,
        "supervised_metrics": sup_metrics,
        "unsupervised_metrics": unsup_metrics,
        "comparison_table": comparison_table,
        "anomaly_examples": anomaly_examples,
        "feature_importance": fi_df,
    }



## Part 8: Script CLI Entry Point (Optional in Notebook)

This section is mainly for `.py` script execution from terminal.
In notebook usage, you usually do not need this part.

In [ ]:
def parse_args() -> PipelineConfig:
    parser = argparse.ArgumentParser(description="Anomaly detection pipeline for slice misrouting.")
    parser.add_argument("--csv-path", type=str, default="network_slicing_dataset - v3.csv")
    parser.add_argument("--output-dir", type=str, default=".")
    parser.add_argument("--inject-synthetic", action="store_true")
    parser.add_argument("--synthetic-fraction", type=float, default=0.20)
    parser.add_argument("--random-seed", type=int, default=RANDOM_SEED)
    parser.add_argument("--drop-impossible-rows", action="store_true")
    parser.add_argument("--keep-impossible-rows", action="store_true")
    parser.add_argument("--severe-latency-ratio", type=float, default=1.5)
    parser.add_argument("--severe-packet-loss-ratio", type=float, default=1.5)
    parser.add_argument("--anomaly-min-violations", type=int, default=2)
    args = parser.parse_args()

    drop_impossible = True
    if args.keep_impossible_rows:
        drop_impossible = False
    elif args.drop_impossible_rows:
        drop_impossible = True

    return PipelineConfig(
        csv_path=args.csv_path,
        output_dir=args.output_dir,
        inject_synthetic=args.inject_synthetic,
        synthetic_fraction=args.synthetic_fraction,
        random_seed=args.random_seed,
        drop_impossible_rows=drop_impossible,
        severe_latency_ratio=args.severe_latency_ratio,
        severe_packet_loss_ratio=args.severe_packet_loss_ratio,
        anomaly_min_violations=args.anomaly_min_violations,
    )


if __name__ == "__main__":
    cfg = parse_args()
    run_pipeline(cfg)


## Part 9: Execute the Pipeline

Edit the config values if needed, then run this cell.

In [ ]:
config = PipelineConfig(
    csv_path='network_slicing_dataset - v3.csv',
    output_dir='.',
    inject_synthetic=True,
    synthetic_fraction=0.20,
    random_seed=42,
    drop_impossible_rows=True,
    severe_latency_ratio=1.5,
    severe_packet_loss_ratio=1.5,
    anomaly_min_violations=2,
)

artifacts = run_pipeline(config)
print('Pipeline complete. Returned keys:', list(artifacts.keys()))
